# Credit Risk Prediction Model

## Objective

The goal of this project is to build a machine learning model that predicts loan default risk using client financial and demographic data.

## Business Problem

Banks need to assess whether a client is likely to default on a loan. Poor credit risk assessment can lead to financial losses.

This project develops predictive models to estimate default probability and support data-driven lending decisions.

## Models Used

- Logistic Regression
- Random Forest

## Expected Outcome

- Predict loan default
- Estimate risk probability
- Identify key risk factors
- Provide business insights

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("credit_risk_dataset.csv", sep=';')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


Source: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
This research employed a binary variable, default payment (Yes = 1, No = 0), as the response variable. This study reviewed the literature and used the following 23 variables as explanatory variables:
X1: Amount of the given credit (NT dollar): it includes both the individual consumer credit and his/her family (supplementary) credit.
X2: Gender (1 = male; 2 = female).
X3: Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).
X4: Marital status (1 = married; 2 = single; 3 = others).
X5: Age (year).
X6 - X11: History of past payment. We tracked the past monthly payment records (from April to September, 2005) as follows: X6 = the repayment status in September, 2005; X7 = the repayment status in August, 2005; . . .;X11 = the repayment status in April, 2005. The measurement scale for the repayment status is: -1 = pay duly; 1 = payment delay for one month; 2 = payment delay for two months; . . .; 8 = payment delay for eight months; 9 = payment delay for nine months and above.
X12-X17: Amount of bill statement (NT dollar). X12 = amount of bill statement in September, 2005; X13 = amount of bill statement in August, 2005; . . .; X17 = amount of bill statement in April, 2005. 
X18-X23: Amount of previous payment (NT dollar). X18 = amount paid in September, 2005; X19 = amount paid in August, 2005; . . .;X23 = amount paid in April, 2005.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [4]:
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [5]:
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [6]:
# individuals with nan at employment length i will replace with zero not drop implying they havent been working
df['person_emp_length'] = df['person_emp_length'].fillna(0)
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length                0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [7]:
# the loan interest rate for some people had NAN, therefore i replaced them with 0 as an assumption that they
# pay zero loan interest
df['loan_int_rate'] = df['loan_int_rate'].fillna(0)
df.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [8]:
# columns sucxh as person home ownership, loan_intent and loan_grade has datatypes: object. 
# this there fore converts them to 0s and 1s for regression analysis

data = pd.get_dummies(df, drop_first=True).copy()
print(data)

       person_age  person_income  person_emp_length  loan_amnt  loan_int_rate  \
0              22          59000              123.0      35000          16.02   
1              21           9600                5.0       1000          11.14   
2              25           9600                1.0       5500          12.87   
3              23          65500                4.0      35000          15.23   
4              24          54400                8.0      35000          14.27   
...           ...            ...                ...        ...            ...   
32576          57          53000                1.0       5800          13.16   
32577          54         120000                4.0      17625           7.49   
32578          65          76000                3.0      35000          10.99   
32579          56         150000                5.0      15000          11.48   
32580          66          42000                2.0       6475           9.99   

       loan_status  loan_pe

In [9]:
#Define features
X = data.drop("loan_status", axis=1)
y = data["loan_status"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [12]:
accuracy_score(y_test, lr_pred)

0.7983734847322388

In [13]:
print(classification_report(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.80      0.98      0.88      5072
           1       0.71      0.15      0.25      1445

    accuracy                           0.80      6517
   macro avg       0.76      0.57      0.57      6517
weighted avg       0.78      0.80      0.74      6517



In [14]:
risk_prob = lr.predict_proba(X_test)[:,1]

In [15]:
results = X_test.copy()

results["Actual"] = y_test
results["Predicted"] = lr_pred
results["Risk_Probability"] = risk_prob

results

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,...,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y,Actual,Predicted,Risk_Probability
14668,24,28000,6.0,10000,10.37,0.36,2,0,1,0,...,1,0,0,0,0,0,0,0,0,0.483289
24614,27,64000,0.0,10000,15.27,0.16,10,0,0,1,...,0,1,0,0,0,0,1,0,0,0.175383
11096,26,72000,10.0,16000,0.00,0.22,3,0,0,0,...,0,0,1,0,0,0,0,0,0,0.226877
10424,23,27996,7.0,10000,0.00,0.36,2,0,0,1,...,0,0,0,0,0,0,0,1,0,0.483329
26007,30,44500,2.0,13000,16.32,0.29,6,0,0,1,...,0,0,0,1,0,0,0,1,0,0.396475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31330,37,88000,20.0,9925,0.00,0.11,16,0,0,0,...,1,0,0,0,0,0,0,0,0,0.072867
2862,26,64000,2.0,3000,0.00,0.05,4,0,0,1,...,0,0,0,0,0,0,0,0,0,0.090504
14754,23,114000,4.0,10000,0.00,0.09,2,0,0,0,...,1,0,0,0,0,0,0,0,0,0.026468
14170,24,100000,1.0,6000,0.00,0.06,3,0,0,0,...,0,0,0,0,0,0,0,0,0,0.030382


In [16]:
rf = RandomForestClassifier()

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [17]:
accuracy_score(y_test, rf_pred)

0.9303360441921129

In [18]:
importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importance.head(25)

loan_percent_income            0.224637
person_income                  0.144418
loan_int_rate                  0.115924
person_home_ownership_RENT     0.079826
loan_amnt                      0.078948
person_emp_length              0.061761
loan_grade_D                   0.055936
person_age                     0.048821
cb_person_cred_hist_length     0.037583
person_home_ownership_OWN      0.017342
loan_grade_C                   0.017151
loan_intent_MEDICAL            0.016119
loan_grade_E                   0.015931
loan_intent_EDUCATION          0.015579
loan_intent_HOMEIMPROVEMENT    0.014938
cb_person_default_on_file_Y    0.013784
loan_intent_PERSONAL           0.013583
loan_intent_VENTURE            0.013307
loan_grade_F                   0.005169
loan_grade_B                   0.005008
loan_grade_G                   0.003161
person_home_ownership_OTHER    0.001075
dtype: float64

In [19]:
print("Logistic Regression accuracy score:", accuracy_score(y_test, lr_pred))
print("Random Forest accuracy score:", accuracy_score(y_test, rf_pred))

Logistic Regression accuracy score: 0.7983734847322388
Random Forest accuracy score: 0.9303360441921129


## Key Findings

Random Forest performed better than Logistic Regression in predicting loan default.

Loan interest rate and loan-to-income ratio were strong predictors of credit risk.

Clients with previous defaults and high loan percentages showed higher default probability.

## Business Impact

This model can help banks:

- assess loan risk
- reduce default rates
- improve credit approval decisions
- increase profitability

save model and feature for streamlit dashboard

In [20]:
import pickle

# save model
with open("credit_model.pkl", "wb") as f:
    pickle.dump(lr, f)

# save feature names
with open("feature_columns.pkl", "wb") as f:
    pickle.dump(X.columns, f)